# Cohort 6: Features Build

In [1]:
%%configure -f
{
    "conf": {
        "spark.broadcast.compress": "true", 
        "spark.jars.packages": "ai.catboost:catboost-spark_3.5_2.12:1.2.7",
        "spark.jars.packages.resolve.transitive": "true",
        "spark.executor.memory": "84g",
        "spark.executor.memoryOverhead": "24g",
        "spark.executor.cores": "1",   
        "spark.executorEnv.CATBOOST_WORKER_INIT_TIMEOUT": "3600s",
        "spark.executor.memoryOverheadFactor": "0.4",
        "spark.executor.extraJavaOptions": "-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35 -XX:ConcGCThreads=2 --add-exports java.base/sun.net.util=ALL-UNNAMED",
        "spark.shuffle.file.buffer": "1m",
        "spark.reducer.maxSizeInFlight": "96m",
        "spark.driver.extraJavaOptions": "--add-exports java.base/sun.net.util=ALL-UNNAMED",
        "spark.driver.memory": "84g",
        "spark.driver.memoryOverhead": "24g",
        "spark.dynamicAllocation.enabled": "true",
        "spark.dynamicAllocation.minExecutors": "4",
        "spark.dynamicAllocation.maxExecutors": "120",
        "spark.memory.fraction": "0.6",
        "spark.memory.storageFraction": "0.4",
        "spark.network.timeout": "1200s",  
        "spark.rpc.askTimeout": "1200s", 
        "spark.rpc.message.maxSize": "512",
        "spark.shuffle.service.enabled": "true",
        "spark.sql.shuffle.partitions": "1000",
        "spark.sql.adaptive.enabled": "true", 
        "spark.sql.broadcastTimeout": "1200s",
        "spark.sql.session.timeout": "1200s",
        "spark.task.cpus": "1",  
        "spark.yarn.am.memory": "12g"
    }
}

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.linalg import Vectors
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
import catboost_spark

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1737331960785_0003,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
# Adding a parameter tag
cohort = 'cohort6'

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# S3 Paths
s3_bucket = f"s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/2_enhanced_datasets/{cohort}"
train_input_path = f"{s3_bucket}/train"
test_input_path = f"{s3_bucket}/test"

# Read processed train and test datasets from S3
print("Reading train and test datasets...")
train_df = spark.read.parquet(train_input_path)
test_df = spark.read.parquet(test_input_path)

print("Train and test datasets successfully loaded.")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reading train and test datasets...
Train and test datasets successfully loaded.

In [5]:
# Verify output
print("Train Dataframe Schema:")
train_df.printSchema()
print("Test Dataframe Schema:")
test_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Train Dataframe Schema:
root
 |-- mi_person_key: string (nullable = true)
 |-- member_age_dos: integer (nullable = true)
 |-- drug_date: date (nullable = true)
 |-- ADE_Date: date (nullable = true)
 |-- standardized_drug_name: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- person_key_index: double (nullable = true)
 |-- drug_name_index: double (nullable = true)
 |-- drug_name_one_hot: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- polypharmacy: long (nullable = true)
 |-- activity_count: long (nullable = true)
 |-- polypharmacy_bin: string (nullable = true)
 |-- activity_count_bin: string (nullable = true)
 |-- activity_tag: string (nullable = true)
 |-- partition_key: integer (nullable = true)

Test Dataframe Schema:
root
 |-- mi_person_key: string (nullable = true)
 |-- member_age_dos: integer (nullable = true)
 |-- drug_date: date (nullable = true)
 |-- ADE_Date: date (nullable = true)
 |-- standardized_drug_name: string (nullable = true)


In [6]:
# Drop the 'features' column from the training DataFrame
train_df = train_df.drop("features")

# Drop the 'features' column from the testing DataFrame
test_df = test_df.drop("features")


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Train Datasets

In [7]:
# Filter by polypharmacy ranges
low_polypharmacy_train = train_df.filter(col("polypharmacy") <= 3)
moderate_polypharmacy_train = train_df.filter((col("polypharmacy") > 3) & (col("polypharmacy") <= 7))
high_polypharmacy_train = train_df.filter(col("polypharmacy") > 7)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Test Datasets

In [8]:
# Filter by polypharmacy ranges
low_polypharmacy_test = test_df.filter(col("polypharmacy") <= 3)
moderate_polypharmacy_test = test_df.filter((col("polypharmacy") > 3) & (col("polypharmacy") <= 7))
high_polypharmacy_test = test_df.filter(col("polypharmacy") > 7)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Cohort 6 Models: Low Polypharmacy

In [9]:
low_polypharmacy_train.schema

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

StructType([StructField('mi_person_key', StringType(), True), StructField('member_age_dos', IntegerType(), True), StructField('drug_date', DateType(), True), StructField('ADE_Date', DateType(), True), StructField('standardized_drug_name', StringType(), True), StructField('label', IntegerType(), True), StructField('person_key_index', DoubleType(), True), StructField('drug_name_index', DoubleType(), True), StructField('drug_name_one_hot', VectorUDT(), True), StructField('polypharmacy', LongType(), True), StructField('activity_count', LongType(), True), StructField('polypharmacy_bin', StringType(), True), StructField('activity_count_bin', StringType(), True), StructField('activity_tag', StringType(), True), StructField('partition_key', IntegerType(), True)])

In [10]:
# Existing Models
import boto3
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# Function to list already saved models in the S3 bucket
def get_existing_models(bucket_name, prefix):
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    existing_models = set()
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                key = obj['Key']
                # Remove the prefix and split remaining path
                relative_path = key[len(prefix):].lstrip('/')
                parts = relative_path.split('/')
                if len(parts) > 0:
                    # Only add the top-level folder name (model identifier)
                    model_identifier = parts[0]
                    existing_models.add(model_identifier)
    return existing_models


# S3 bucket and prefix
bucket_name = "pgx-repository"
prefix = f"ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}"

# Fetch existing models from S3
existing_models = get_existing_models(bucket_name, prefix)

sorted_models = sorted(existing_models, key=lambda x: int(x.split('_')[-1]))
print(sorted_models)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['spark_model_1000', 'spark_model_1001', 'spark_model_1002', 'spark_model_1003', 'spark_model_1004', 'spark_model_1005', 'spark_model_1006', 'spark_model_1007', 'spark_model_1008', 'spark_model_1009', 'spark_model_3000', 'spark_model_3001', 'spark_model_3002', 'spark_model_3003', 'spark_model_3004', 'spark_model_3005', 'spark_model_3006', 'spark_model_3007']

In [11]:
# Model Features
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=["person_key_index", "drug_name_index"],  # Include mi_person_key as an indexed categorical feature
    outputCol="features"
)

# Create pipeline
pipeline = Pipeline(stages=[assembler])

# Fit and transform the data
pipeline_model = pipeline.fit(train_df)

# Transform the dataset
train_low = pipeline_model.transform(low_polypharmacy_train)
test_low = pipeline_model.transform(low_polypharmacy_test)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
# CatBoost Pool objects
from pyspark import StorageLevel

# Cache/persist the Spark DataFrames before creating the Pool
train_df = train_low.select("features", "label").persist(StorageLevel.MEMORY_AND_DISK)
test_df = test_low.select("features", "label").persist(StorageLevel.MEMORY_AND_DISK)

# Create the CatBoost Pool objects
train_pool = catboost_spark.Pool(train_df)
test_pool = catboost_spark.Pool(test_df)

# Confirm the DataFrames are cached/persisted
print(train_df.storageLevel)
print(test_df.storageLevel)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Disk Memory Serialized 1x Replicated
Disk Memory Serialized 1x Replicated

In [13]:
# Build Models..
import io
import pandas as pd
import boto3

# Initialize boto3 S3 client
s3_client = boto3.client('s3')

# Seeds for different runs - 10 models
seeds = [3, 24, 18, 17, 19, 11, 38, 74, 35, 90]

# Start model number tracker
model_series = 1000

# Loop to train and save models (10 runs for stable feature selection)
for seed in seeds:
    # Check if the model already exists in S3
    model_key = f"spark_model_{model_series}"
    if model_key in existing_models:
        print(f"Model {model_key} already exists in S3. Skipping...")
        model_series += 1
        continue

    print(f"Training model {model_series} with seed {seed}...")
    
    # Initialize CatBoost Spark Classifier with the current seed
    classifier = catboost_spark.CatBoostClassifier(randomSeed=seed)

    # Train the model
    model = classifier.fit(train_pool, evalDatasets=[test_pool])

    # Define the path to save the Spark model, including the model number
    spark_model_path = f"s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}/spark_model_{model_series}"

    # Save the Spark model (with metadata)
    model.write().overwrite().save(spark_model_path)
    print(f"Spark model {model_series} with metadata saved to: {spark_model_path}")
    
    # Clean up memory for next run
    del classifier
    del model
    
    # Increment the model number for the next run
    model_series += 1


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model spark_model_1000 already exists in S3. Skipping...
Model spark_model_1001 already exists in S3. Skipping...
Model spark_model_1002 already exists in S3. Skipping...
Model spark_model_1003 already exists in S3. Skipping...
Model spark_model_1004 already exists in S3. Skipping...
Model spark_model_1005 already exists in S3. Skipping...
Model spark_model_1006 already exists in S3. Skipping...
Model spark_model_1007 already exists in S3. Skipping...
Model spark_model_1008 already exists in S3. Skipping...
Model spark_model_1009 already exists in S3. Skipping...

# Cohort 6 Models: Moderate Polypharmacy

In [14]:
# Existing Models
import boto3
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# Function to list already saved models in the S3 bucket
def get_existing_models(bucket_name, prefix):
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    existing_models = set()
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                key = obj['Key']
                # Remove the prefix and split remaining path
                relative_path = key[len(prefix):].lstrip('/')
                parts = relative_path.split('/')
                if len(parts) > 0:
                    # Only add the top-level folder name (model identifier)
                    model_identifier = parts[0]
                    existing_models.add(model_identifier)
    return existing_models


# S3 bucket and prefix
bucket_name = "pgx-repository"
prefix = f"ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}"

# Fetch existing models from S3
existing_models = get_existing_models(bucket_name, prefix)

sorted_models = sorted(existing_models, key=lambda x: int(x.split('_')[-1]))
print(sorted_models)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['spark_model_1000', 'spark_model_1001', 'spark_model_1002', 'spark_model_1003', 'spark_model_1004', 'spark_model_1005', 'spark_model_1006', 'spark_model_1007', 'spark_model_1008', 'spark_model_1009', 'spark_model_3000', 'spark_model_3001', 'spark_model_3002', 'spark_model_3003', 'spark_model_3004', 'spark_model_3005', 'spark_model_3006', 'spark_model_3007']

In [15]:
moderate_polypharmacy_train.schema

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

StructType([StructField('mi_person_key', StringType(), True), StructField('member_age_dos', IntegerType(), True), StructField('drug_date', DateType(), True), StructField('ADE_Date', DateType(), True), StructField('standardized_drug_name', StringType(), True), StructField('label', IntegerType(), True), StructField('person_key_index', DoubleType(), True), StructField('drug_name_index', DoubleType(), True), StructField('drug_name_one_hot', VectorUDT(), True), StructField('polypharmacy', LongType(), True), StructField('activity_count', LongType(), True), StructField('polypharmacy_bin', StringType(), True), StructField('activity_count_bin', StringType(), True), StructField('activity_tag', StringType(), True), StructField('partition_key', IntegerType(), True)])

# Cohort 6 Models: Low Polypharmacy

In [9]:
low_polypharmacy_train.schema

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

StructType([StructField('mi_person_key', StringType(), True), StructField('member_age_dos', IntegerType(), True), StructField('drug_date', DateType(), True), StructField('ADE_Date', DateType(), True), StructField('standardized_drug_name', StringType(), True), StructField('label', IntegerType(), True), StructField('person_key_index', DoubleType(), True), StructField('drug_name_index', DoubleType(), True), StructField('drug_name_one_hot', VectorUDT(), True), StructField('polypharmacy', LongType(), True), StructField('activity_count', LongType(), True), StructField('polypharmacy_bin', StringType(), True), StructField('activity_count_bin', StringType(), True), StructField('activity_tag', StringType(), True), StructField('partition_key', IntegerType(), True)])

In [10]:
# Existing Models
import boto3
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# Function to list already saved models in the S3 bucket
def get_existing_models(bucket_name, prefix):
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    existing_models = set()
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                key = obj['Key']
                # Remove the prefix and split remaining path
                relative_path = key[len(prefix):].lstrip('/')
                parts = relative_path.split('/')
                if len(parts) > 0:
                    # Only add the top-level folder name (model identifier)
                    model_identifier = parts[0]
                    existing_models.add(model_identifier)
    return existing_models


# S3 bucket and prefix
bucket_name = "pgx-repository"
prefix = f"ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}"

# Fetch existing models from S3
existing_models = get_existing_models(bucket_name, prefix)

sorted_models = sorted(existing_models, key=lambda x: int(x.split('_')[-1]))
print(sorted_models)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['spark_model_1000', 'spark_model_1001', 'spark_model_1002', 'spark_model_1003', 'spark_model_1004', 'spark_model_1005', 'spark_model_1006', 'spark_model_1007', 'spark_model_1008', 'spark_model_1009', 'spark_model_3000', 'spark_model_3001', 'spark_model_3002', 'spark_model_3003', 'spark_model_3004', 'spark_model_3005', 'spark_model_3006', 'spark_model_3007']

In [11]:
# Model Features
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=["person_key_index", "drug_name_index"],  # Include mi_person_key as an indexed categorical feature
    outputCol="features"
)

# Create pipeline
pipeline = Pipeline(stages=[assembler])

# Fit and transform the data
pipeline_model = pipeline.fit(train_df)

# Transform the dataset
train_low = pipeline_model.transform(low_polypharmacy_train)
test_low = pipeline_model.transform(low_polypharmacy_test)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
# CatBoost Pool objects
from pyspark import StorageLevel

# Cache/persist the Spark DataFrames before creating the Pool
train_df = train_low.select("features", "label").persist(StorageLevel.MEMORY_AND_DISK)
test_df = test_low.select("features", "label").persist(StorageLevel.MEMORY_AND_DISK)

# Create the CatBoost Pool objects
train_pool = catboost_spark.Pool(train_df)
test_pool = catboost_spark.Pool(test_df)

# Confirm the DataFrames are cached/persisted
print(train_df.storageLevel)
print(test_df.storageLevel)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Disk Memory Serialized 1x Replicated
Disk Memory Serialized 1x Replicated

In [13]:
# Build Models..
import io
import pandas as pd
import boto3

# Initialize boto3 S3 client
s3_client = boto3.client('s3')

# Seeds for different runs - 10 models
seeds = [3, 24, 18, 17, 19, 11, 38, 74, 35, 90]

# Start model number tracker
model_series = 1000

# Loop to train and save models (10 runs for stable feature selection)
for seed in seeds:
    # Check if the model already exists in S3
    model_key = f"spark_model_{model_series}"
    if model_key in existing_models:
        print(f"Model {model_key} already exists in S3. Skipping...")
        model_series += 1
        continue

    print(f"Training model {model_series} with seed {seed}...")
    
    # Initialize CatBoost Spark Classifier with the current seed
    classifier = catboost_spark.CatBoostClassifier(randomSeed=seed)

    # Train the model
    model = classifier.fit(train_pool, evalDatasets=[test_pool])

    # Define the path to save the Spark model, including the model number
    spark_model_path = f"s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}/spark_model_{model_series}"

    # Save the Spark model (with metadata)
    model.write().overwrite().save(spark_model_path)
    print(f"Spark model {model_series} with metadata saved to: {spark_model_path}")
    
    # Clean up memory for next run
    del classifier
    del model
    
    # Increment the model number for the next run
    model_series += 1


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model spark_model_1000 already exists in S3. Skipping...
Model spark_model_1001 already exists in S3. Skipping...
Model spark_model_1002 already exists in S3. Skipping...
Model spark_model_1003 already exists in S3. Skipping...
Model spark_model_1004 already exists in S3. Skipping...
Model spark_model_1005 already exists in S3. Skipping...
Model spark_model_1006 already exists in S3. Skipping...
Model spark_model_1007 already exists in S3. Skipping...
Model spark_model_1008 already exists in S3. Skipping...
Model spark_model_1009 already exists in S3. Skipping...

# Cohort 6 Models: Moderate Polypharmacy

In [14]:
# Existing Models
import boto3
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# Function to list already saved models in the S3 bucket
def get_existing_models(bucket_name, prefix):
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    existing_models = set()
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                key = obj['Key']
                # Remove the prefix and split remaining path
                relative_path = key[len(prefix):].lstrip('/')
                parts = relative_path.split('/')
                if len(parts) > 0:
                    # Only add the top-level folder name (model identifier)
                    model_identifier = parts[0]
                    existing_models.add(model_identifier)
    return existing_models


# S3 bucket and prefix
bucket_name = "pgx-repository"
prefix = f"ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}"

# Fetch existing models from S3
existing_models = get_existing_models(bucket_name, prefix)

sorted_models = sorted(existing_models, key=lambda x: int(x.split('_')[-1]))
print(sorted_models)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['spark_model_1000', 'spark_model_1001', 'spark_model_1002', 'spark_model_1003', 'spark_model_1004', 'spark_model_1005', 'spark_model_1006', 'spark_model_1007', 'spark_model_1008', 'spark_model_1009', 'spark_model_3000', 'spark_model_3001', 'spark_model_3002', 'spark_model_3003', 'spark_model_3004', 'spark_model_3005', 'spark_model_3006', 'spark_model_3007']

In [15]:
moderate_polypharmacy_train.schema

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

StructType([StructField('mi_person_key', StringType(), True), StructField('member_age_dos', IntegerType(), True), StructField('drug_date', DateType(), True), StructField('ADE_Date', DateType(), True), StructField('standardized_drug_name', StringType(), True), StructField('label', IntegerType(), True), StructField('person_key_index', DoubleType(), True), StructField('drug_name_index', DoubleType(), True), StructField('drug_name_one_hot', VectorUDT(), True), StructField('polypharmacy', LongType(), True), StructField('activity_count', LongType(), True), StructField('polypharmacy_bin', StringType(), True), StructField('activity_count_bin', StringType(), True), StructField('activity_tag', StringType(), True), StructField('partition_key', IntegerType(), True)])

In [17]:
# Model Features
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=["person_key_index", "drug_name_index"],  # Include mi_person_key as an indexed categorical feature
    outputCol="features"
)

# Create pipeline
pipeline = Pipeline(stages=[assembler])

# Fit and transform the data
pipeline_model = pipeline.fit(moderate_polypharmacy_train)

# Transform the dataset
train_mid = pipeline_model.transform(moderate_polypharmacy_train)
test_mid = pipeline_model.transform(moderate_polypharmacy_test)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
# CatBoost Pool objects
from pyspark import StorageLevel

# Cache/persist the Spark DataFrames before creating the Pool
train_df = train_mid.select("features", "label").persist(StorageLevel.MEMORY_AND_DISK)
test_df = test_mid.select("features", "label").persist(StorageLevel.MEMORY_AND_DISK)

# Create the CatBoost Pool objects
train_pool = catboost_spark.Pool(train_df)
test_pool = catboost_spark.Pool(test_df)

# Confirm the DataFrames are cached/persisted
print(train_df.storageLevel)
print(test_df.storageLevel)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Disk Memory Serialized 1x Replicated
Disk Memory Serialized 1x Replicated

In [20]:
# Build Models..
import io
import pandas as pd
import boto3

# Initialize boto3 S3 client
s3_client = boto3.client('s3')

# Seeds for different runs - 10 models
seeds = [3, 24, 18, 17, 19, 11, 38, 74, 35, 90]

# Start model number tracker
model_series = 3000

# Loop to train and save models (10 runs for stable feature selection)
for seed in seeds:
    # Check if the model already exists in S3
    model_key = f"spark_model_{model_series}"
    if model_key in existing_models:
        print(f"Model {model_key} already exists in S3. Skipping...")
        model_series += 1
        continue

    print(f"Training model {model_series} with seed {seed}...")
    
    # Initialize CatBoost Spark Classifier with the current seed
    classifier = catboost_spark.CatBoostClassifier(randomSeed=seed)

    # Train the model
    model = classifier.fit(train_pool, evalDatasets=[test_pool])

    # Define the path to save the Spark model, including the model number
    spark_model_path = f"s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}/spark_model_{model_series}"

    # Save the Spark model (with metadata)
    model.write().overwrite().save(spark_model_path)
    print(f"Spark model {model_series} with metadata saved to: {spark_model_path}")
    
    # Clean up memory for next run
    del classifier
    del model
    
    # Increment the model number for the next run
    model_series += 1


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model spark_model_3000 already exists in S3. Skipping...
Model spark_model_3001 already exists in S3. Skipping...
Model spark_model_3002 already exists in S3. Skipping...
Model spark_model_3003 already exists in S3. Skipping...
Model spark_model_3004 already exists in S3. Skipping...
Model spark_model_3005 already exists in S3. Skipping...
Model spark_model_3006 already exists in S3. Skipping...
Model spark_model_3007 already exists in S3. Skipping...
Training model 3008 with seed 35...
Spark model 3008 with metadata saved to: s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/4_models/cohort6/spark_model_3008
Training model 3009 with seed 90...
Spark model 3009 with metadata saved to: s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/4_models/cohort6/spark_model_3009

# Cohort 6 Models: High Polypharmacy

In [21]:
# Existing Models
import boto3
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# Function to list already saved models in the S3 bucket
def get_existing_models(bucket_name, prefix):
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    existing_models = set()
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                key = obj['Key']
                # Remove the prefix and split remaining path
                relative_path = key[len(prefix):].lstrip('/')
                parts = relative_path.split('/')
                if len(parts) > 0:
                    # Only add the top-level folder name (model identifier)
                    model_identifier = parts[0]
                    existing_models.add(model_identifier)
    return existing_models


# S3 bucket and prefix
bucket_name = "pgx-repository"
prefix = f"ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}"

# Fetch existing models from S3
existing_models = get_existing_models(bucket_name, prefix)

sorted_models = sorted(existing_models, key=lambda x: int(x.split('_')[-1]))
print(sorted_models)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['spark_model_1000', 'spark_model_1001', 'spark_model_1002', 'spark_model_1003', 'spark_model_1004', 'spark_model_1005', 'spark_model_1006', 'spark_model_1007', 'spark_model_1008', 'spark_model_1009', 'spark_model_3000', 'spark_model_3001', 'spark_model_3002', 'spark_model_3003', 'spark_model_3004', 'spark_model_3005', 'spark_model_3006', 'spark_model_3007', 'spark_model_3008', 'spark_model_3009']

In [22]:
high_polypharmacy_train.schema

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

StructType([StructField('mi_person_key', StringType(), True), StructField('member_age_dos', IntegerType(), True), StructField('drug_date', DateType(), True), StructField('ADE_Date', DateType(), True), StructField('standardized_drug_name', StringType(), True), StructField('label', IntegerType(), True), StructField('person_key_index', DoubleType(), True), StructField('drug_name_index', DoubleType(), True), StructField('drug_name_one_hot', VectorUDT(), True), StructField('polypharmacy', LongType(), True), StructField('activity_count', LongType(), True), StructField('polypharmacy_bin', StringType(), True), StructField('activity_count_bin', StringType(), True), StructField('activity_tag', StringType(), True), StructField('partition_key', IntegerType(), True)])

In [23]:
# Add model features
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=["person_key_index", "drug_name_index"],  # Include mi_person_key as an indexed categorical feature
    outputCol="features"
)

# Create pipeline
pipeline = Pipeline(stages=[assembler])

# Fit and transform the data
pipeline_model = pipeline.fit(high_polypharmacy_train)

# Transform the dataset
train_high = pipeline_model.transform(high_polypharmacy_train)
test_high = pipeline_model.transform(high_polypharmacy_test)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [24]:
# CatBoost Pool objects
from pyspark import StorageLevel

# Cache or persist the Spark DataFrames before creating the Pool
train_df = train_high.select("features", "label").persist(StorageLevel.MEMORY_AND_DISK)
test_df = test_high.select("features", "label").persist(StorageLevel.MEMORY_AND_DISK)

# Create the CatBoost Pool objects
train_pool = catboost_spark.Pool(train_df)
test_pool = catboost_spark.Pool(test_df)

# Confirm the DataFrames are cached/persisted
print(train_df.storageLevel)
print(test_df.storageLevel)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Disk Memory Serialized 1x Replicated
Disk Memory Serialized 1x Replicated

In [25]:
# Build Models..
import io
import pandas as pd
import boto3

# Initialize boto3 S3 client
s3_client = boto3.client('s3')

# Seeds for different runs - 10 models
seeds = [3, 24, 18, 17, 19, 11, 38, 74, 35, 90]

# Start model number tracker
model_series = 9000

# Loop to train and save models (10 runs for stable feature selection)
for seed in seeds:
    # Check if the model already exists in S3
    model_key = f"spark_model_{model_series}"
    if model_key in existing_models:
        print(f"Model {model_key} already exists in S3. Skipping...")
        model_series += 1
        continue

    print(f"Training model {model_series} with seed {seed}...")
    
    # Initialize CatBoost Spark Classifier with the current seed
    classifier = catboost_spark.CatBoostClassifier(randomSeed=seed)

    # Train the model
    model = classifier.fit(train_pool, evalDatasets=[test_pool])

    # Define the path to save the Spark model, including the model number
    spark_model_path = f"s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}/spark_model_{model_series}"

    # Save the Spark model (with metadata)
    model.write().overwrite().save(spark_model_path)
    print(f"Spark model {model_series} with metadata saved to: {spark_model_path}")
    
    # Clean up memory for next run
    del classifier
    del model
    
    # Increment the model number for the next run
    model_series += 1


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
capacity < 0: (-100663296 < 0)
Traceback (most recent call last):
  File "/mnt/yarn/usercache/livy/appcache/application_1737331960785_0003/container_1737331960785_0003_01_000001/ai.catboost_catboost-spark_3.5_2.12-1.2.7.jar/catboost_spark/core.py", line 5362, in fit
    return _fit_with_eval(params)
  File "/mnt/yarn/usercache/livy/appcache/application_1737331960785_0003/container_1737331960785_0003_01_000001/ai.catboost_catboost-spark_3.5_2.12-1.2.7.jar/catboost_spark/core.py", line 5359, in _fit_with_eval
    return self._fit_with_eval(trainDatasetAsJavaObject, evalDatasetsAsJavaObject, params)
  File "/mnt/yarn/usercache/livy/appcache/application_1737331960785_0003/container_1737331960785_0003_01_000001/ai.catboost_catboost-spark_3.5_2.12-1.2.7.jar/catboost_spark/core.py", line 5316, in _fit_with_eval
    java_model = self._java_obj.fit(trainDatasetAsJavaObject, evalDatasetsAsJavaObject)
  File "/mnt/yarn/usercache/livy/appcache/application_1737331960785_00